In [1]:
from pathlib import Path
from dotenv import load_dotenv

raiz_projeto = Path.cwd()
while not (raiz_projeto / ".env.example").is_file() and raiz_projeto.parent != raiz_projeto:
    raiz_projeto = raiz_projeto.parent

caminho_env = raiz_projeto / ".env"
if not caminho_env.is_file():
    raise FileNotFoundError(
        f"Arquivo .env não encontrado em {raiz_projeto}. "
        "Copie .env.example para .env e preencha as configurações localmente."
    )

load_dotenv(caminho_env, override=False)
print("Configuração local do Snowflake carregada.")

Configuração local do Snowflake carregada.


In [2]:
import csv
import hashlib
import re
from collections import Counter
from pathlib import Path

ANOS_ESPERADOS = list(range(2015, 2025))
PASTA_BASE = raiz_projeto / "dados" / "bruto"

if not PASTA_BASE.is_dir():
    raise FileNotFoundError(f"Pasta de dados não encontrada: {PASTA_BASE}")


def normalizar_coluna(nome):
    return nome.strip().lstrip("\ufeff").upper()


def inspecionar_arquivos(config):
    encontrados = []
    padrao = re.compile(config["filename_regex"], re.IGNORECASE)

    for caminho in sorted(Path(config["base_dir"]).rglob("*")):
        if not caminho.is_file():
            continue

        correspondencia = padrao.fullmatch(caminho.name)
        if not correspondencia:
            continue

        ano = int(correspondencia.group(1))
        if ano not in config["years"]:
            continue

        with caminho.open(encoding=config["encoding"], newline="") as arquivo:
            cabecalho = next(csv.reader(arquivo, delimiter=config["delimiter"]), None)

        if not cabecalho:
            raise ValueError(f"CSV sem cabeçalho: {caminho}")

        colunas = [normalizar_coluna(nome) for nome in cabecalho]
        if len(colunas) != len(set(colunas)):
            raise ValueError(f"Colunas duplicadas após normalização: {caminho}")
        if any(not coluna or coluna.startswith("ETL_") for coluna in colunas):
            raise ValueError(f"Cabeçalho vazio ou coluna ETL reservada: {caminho}")

        encontrados.append({
            "path": str(caminho),
            "year": ano,
            "columns": colunas,
        })

    contagens = Counter(arquivo["year"] for arquivo in encontrados)
    invalidos = {
        ano: contagens.get(ano, 0)
        for ano in config["years"]
        if contagens.get(ano, 0) != 1
    }
    if invalidos:
        raise ValueError(
            f"Esperado exatamente um CSV por ano para {config['cadastro']}; "
            f"contagens encontradas: {invalidos}"
        )

    return sorted(encontrados, key=lambda arquivo: arquivo["year"])


def rows(arquivo, config, union):
    with open(arquivo["path"], encoding=config["encoding"], newline="") as stream:
        leitor = csv.reader(stream, delimiter=config["delimiter"], strict=True)
        next(leitor, None)

        for linha in leitor:
            if len(linha) != len(arquivo["columns"]):
                raise ValueError(
                    f"{arquivo['path']}, linha {leitor.line_num}: largura inválida"
                )

            valores = {
                coluna: valor if valor.strip() else None
                for coluna, valor in zip(arquivo["columns"], linha)
            }
            ano_csv = valores.get("NU_ANO_CENSO")
            if ano_csv is not None and ano_csv != str(arquivo["year"]):
                raise ValueError(
                    f"Ano divergente em {arquivo['path']}, linha {leitor.line_num}"
                )

            yield {coluna: valores.get(coluna) for coluna in union}


def checksum(caminho):
    sha256 = hashlib.sha256()
    with open(caminho, "rb") as stream:
        for bloco in iter(lambda: stream.read(1024 * 1024), b""):
            sha256.update(bloco)
    return sha256.hexdigest()


configuracoes = {}
for cadastro in ("CURSOS", "IES"):
    configuracoes[cadastro] = {
        "cadastro": cadastro,
        "base_dir": str(PASTA_BASE),
        "filename_regex": rf"MICRODADOS_CADASTRO_{cadastro}_(20\d{{2}})\.CSV",
        "years": ANOS_ESPERADOS,
        "encoding": "latin1",
        "delimiter": ";",
        "output_dir": raiz_projeto / "reports" / cadastro.lower(),
        "table": f"STG_{cadastro}",
    }

inventarios = {
    cadastro: inspecionar_arquivos(config)
    for cadastro, config in configuracoes.items()
}
schemas = {
    cadastro: sorted({
        coluna
        for arquivo in inventarios[cadastro]
        for coluna in arquivo["columns"]
    })
    for cadastro in configuracoes
}

for cadastro in configuracoes:
    print(
        f"{cadastro}: {len(inventarios[cadastro])} arquivos; "
        f"{len(schemas[cadastro])} colunas no schema"
    )

CURSOS: 10 arquivos; 227 colunas no schema
IES: 10 arquivos; 87 colunas no schema


In [3]:
# ============================================================
# CARGA DOS MICRODADOS INEP PARA SNOWFLAKE
# ============================================================

import os
import re
import json
import hashlib

from pathlib import Path
from datetime import datetime, timezone
from itertools import islice

import pandas as pd

import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

from dotenv import load_dotenv


# ============================================================
# 1. CARREGAR VARIÁVEIS DE AMBIENTE
# ============================================================

load_dotenv()


SF_ACCOUNT = os.getenv("SNOWFLAKE_ACCOUNT")
SF_USER = os.getenv("SNOWFLAKE_USER")
SF_PASSWORD = os.getenv("SNOWFLAKE_PASSWORD")

SF_WAREHOUSE = os.getenv("SNOWFLAKE_WAREHOUSE")
SF_DATABASE = os.getenv("SNOWFLAKE_DATABASE")
SF_SCHEMA = os.getenv("SNOWFLAKE_SCHEMA")
SF_ROLE = os.getenv("SNOWFLAKE_ROLE")


# ============================================================
# 2. VALIDAR CONFIGURAÇÕES
# ============================================================

variaveis_obrigatorias = {
    "SNOWFLAKE_ACCOUNT": SF_ACCOUNT,
    "SNOWFLAKE_USER": SF_USER,
    "SNOWFLAKE_PASSWORD": SF_PASSWORD,
    "SNOWFLAKE_WAREHOUSE": SF_WAREHOUSE,
    "SNOWFLAKE_DATABASE": SF_DATABASE,
    "SNOWFLAKE_SCHEMA": SF_SCHEMA,
    "SNOWFLAKE_ROLE": SF_ROLE,
}


faltantes = [
    nome
    for nome, valor in variaveis_obrigatorias.items()
    if not valor
]


if faltantes:
    raise ValueError(
        "Variáveis não configuradas no .env:\n"
        + "\n".join(faltantes)
    )


print("Configurações do Snowflake carregadas.")


# ============================================================
# 3. CONEXÃO COM SNOWFLAKE
# ============================================================

def conectar_snowflake():

    conn = snowflake.connector.connect(
        account=SF_ACCOUNT,
        user=SF_USER,
        password=SF_PASSWORD,
        warehouse=SF_WAREHOUSE,
        database=SF_DATABASE,
        schema=SF_SCHEMA,
        role=SF_ROLE,
        autocommit=True
    )

    return conn


conn_sf = conectar_snowflake()


print("Conectado ao Snowflake.")


# ============================================================
# 4. TESTAR A CONEXÃO
# ============================================================

with conn_sf.cursor() as cursor:

    cursor.execute("""
        SELECT
            CURRENT_DATABASE(),
            CURRENT_SCHEMA(),
            CURRENT_WAREHOUSE(),
            CURRENT_ROLE(),
            CURRENT_USER()
    """)

    resultado = cursor.fetchone()


print(
    f"""
Conexão Snowflake OK

Database  : {resultado[0]}
Schema    : {resultado[1]}
Warehouse : {resultado[2]}
Role      : {resultado[3]}
User      : {resultado[4]}
"""
)


# ============================================================
# 5. FUNÇÕES AUXILIARES
# ============================================================

def quote_identifier(nome):
    """
    Protege nomes de tabelas e colunas no Snowflake.
    """

    nome = str(nome)

    nome = nome.replace('"', '""')

    return f'"{nome}"'


def tabela_completa(table_name):

    return (
        f"{quote_identifier(SF_DATABASE)}."
        f"{quote_identifier(SF_SCHEMA)}."
        f"{quote_identifier(table_name)}"
    )


# ============================================================
# 6. CRIAR TABELA STAGING
# ============================================================

def criar_tabela_snowflake(
    conn,
    table_name,
    union
):

    """
    Cria a tabela STG no Snowflake.

    As colunas originais do INEP são carregadas inicialmente
    como VARCHAR.

    O tratamento de tipos pode ser realizado posteriormente
    em uma VIEW ou tabela tratada.
    """

    if not re.fullmatch(
        r"[A-Za-z_][A-Za-z0-9_]{0,127}",
        table_name
    ):
        raise ValueError(
            f"Nome de tabela inválido: {table_name}"
        )

    # --------------------------------------------------------
    # Colunas provenientes do CSV
    # --------------------------------------------------------

    colunas_csv = []

    for coluna in union:

        coluna_sql = quote_identifier(coluna)

        colunas_csv.append(
            f"{coluna_sql} VARCHAR"
        )

    # --------------------------------------------------------
    # Metadados ETL
    # --------------------------------------------------------

    colunas_etl = [

        '"ETL_ANO" NUMBER(4,0) NOT NULL',

        '"ETL_ARQUIVO" VARCHAR',

        '"ETL_SHA256" VARCHAR(64)',

        '"ETL_CARGA_UTC" TIMESTAMP_NTZ'
    ]

    todas_colunas = (
        colunas_csv
        +
        colunas_etl
    )

    ddl = f"""
        CREATE TABLE IF NOT EXISTS
        {tabela_completa(table_name)}
        (
            {", ".join(todas_colunas)}
        )
    """

    with conn.cursor() as cursor:

        cursor.execute(ddl)

    print(
        f"Tabela verificada/criada: "
        f"{SF_DATABASE}.{SF_SCHEMA}.{table_name}"
    )


# ============================================================
# 7. GERADOR DE BATCHES
# ============================================================

def gerar_batches(
    iterador,
    tamanho=50000
):

    """
    Divide o gerador de registros em blocos para evitar
    carregar todo o arquivo na memória.
    """

    while True:

        batch = list(
            islice(
                iterador,
                tamanho
            )
        )

        if not batch:
            break

        yield batch


# ============================================================
# 8. NORMALIZAR DATAFRAME PARA O SNOWFLAKE
# ============================================================

def preparar_dataframe(
    batch,
    ano,
    arquivo,
    digest,
    stamp
):

    df = pd.DataFrame(batch)

    # --------------------------------------------------------
    # Metadata ETL
    # --------------------------------------------------------

    df["ETL_ANO"] = ano

    df["ETL_ARQUIVO"] = str(
        arquivo
    )

    df["ETL_SHA256"] = digest

    df["ETL_CARGA_UTC"] = stamp

    # --------------------------------------------------------
    # Substituir NaN por None
    # --------------------------------------------------------

    df = df.where(
        pd.notnull(df),
        None
    )

    return df


# ============================================================
# 9. CRIAR TABELA TEMPORÁRIA
# ============================================================

def criar_tabela_temporaria(
    conn,
    table_name,
    temp_table
):

    sql = f"""
        CREATE OR REPLACE TEMPORARY TABLE
        {quote_identifier(temp_table)}
        LIKE
        {tabela_completa(table_name)}
    """

    with conn.cursor() as cursor:

        cursor.execute(sql)


# ============================================================
# 10. CARREGAR DATAFRAME NA TEMP
# ============================================================

def escrever_dataframe_snowflake(
    conn,
    df,
    temp_table,
    chunk_size=50000
):

    resultado = write_pandas(

        conn=conn,

        df=df,

        table_name=temp_table,

        database=SF_DATABASE,

        schema=SF_SCHEMA,

        chunk_size=chunk_size,

        compression="snappy",

        quote_identifiers=True,

        auto_create_table=False,

        overwrite=False
    )

    success, nchunks, nrows, output = resultado

    if not success:

        raise RuntimeError(
            f"""
            Erro durante o write_pandas.

            Tabela:
            {temp_table}
            """
        )

    return nrows


# ============================================================
# 11. VALIDAR TABELA TEMPORÁRIA
# ============================================================

def contar_temp(
    conn,
    temp_table
):

    sql = f"""
        SELECT COUNT(*)
        FROM {quote_identifier(temp_table)}
    """

    with conn.cursor() as cursor:

        cursor.execute(sql)

        quantidade = cursor.fetchone()[0]

    return quantidade


# ============================================================
# 12. PUBLICAR DADOS DA TEMP NA STG
# ============================================================

def publicar_ano(
    conn,
    table_name,
    temp_table,
    ano
):

    """
    Só substitui os dados do ano depois que a carga
    temporária foi concluída com sucesso.
    """

    target = tabela_completa(
        table_name
    )

    temp = quote_identifier(
        temp_table
    )

    cursor = conn.cursor()

    try:

        # ----------------------------------------------------
        # Iniciar transação
        # ----------------------------------------------------

        cursor.execute(
            "BEGIN"
        )

        # ----------------------------------------------------
        # Remover versão anterior do ano
        # ----------------------------------------------------

        cursor.execute(
            f"""
            DELETE FROM {target}
            WHERE "ETL_ANO" = %s
            """,
            (ano,)
        )

        # ----------------------------------------------------
        # Inserir nova versão
        # ----------------------------------------------------

        cursor.execute(
            f"""
            INSERT INTO {target}
            SELECT *
            FROM {temp}
            """
        )

        # ----------------------------------------------------
        # Commit
        # ----------------------------------------------------

        cursor.execute(
            "COMMIT"
        )

    except Exception:

        cursor.execute(
            "ROLLBACK"
        )

        raise

    finally:

        cursor.close()


# ============================================================
# 13. VALIDAR CARGA FINAL
# ============================================================

def validar_carga_final(
    conn,
    table_name,
    ano
):

    sql = f"""
        SELECT COUNT(*)
        FROM {tabela_completa(table_name)}
        WHERE "ETL_ANO" = %s
    """

    with conn.cursor() as cursor:

        cursor.execute(
            sql,
            (ano,)
        )

        quantidade = cursor.fetchone()[0]

    return quantidade


# ============================================================
# 14. FUNÇÃO PRINCIPAL DE CARGA
# ============================================================

def load_snowflake(
    conn,
    files,
    cfg,
    union,
    output,
    batch_size=50000
):

    """
    Pipeline completo de carga:

    CSV
      ↓
    rows()
      ↓
    batches
      ↓
    DataFrame
      ↓
    write_pandas
      ↓
    TEMP TABLE
      ↓
    validação
      ↓
    DELETE ano anterior
      ↓
    INSERT nova versão
      ↓
    STG
    """

    table_name = cfg["table"]

    # --------------------------------------------------------
    # Criar tabela definitiva
    # --------------------------------------------------------

    criar_tabela_snowflake(
        conn,
        table_name,
        union
    )

    # ========================================================
    # PROCESSAR CADA ARQUIVO / ANO
    # ========================================================

    for file in files:

        ano = file["year"]

        caminho = Path(
            file["path"]
        )

        print()
        print("=" * 70)
        print(
            f"INICIANDO CARGA"
        )
        print(
            f"Tabela : {table_name}"
        )
        print(
            f"Ano    : {ano}"
        )
        print(
            f"Arquivo: {caminho.name}"
        )
        print("=" * 70)

        # ----------------------------------------------------
        # Checksum
        # ----------------------------------------------------

        digest = checksum(
            caminho
        )

        # ----------------------------------------------------
        # Data/hora carga
        # ----------------------------------------------------

        stamp = datetime.now(
            timezone.utc
        ).replace(
            tzinfo=None
        )

        # ----------------------------------------------------
        # Nome temp
        # ----------------------------------------------------

        temp_table = (
            f"TMP_{table_name}_{ano}"
        )

        # ----------------------------------------------------
        # Criar temp
        # ----------------------------------------------------

        criar_tabela_temporaria(
            conn,
            table_name,
            temp_table
        )

        # ----------------------------------------------------
        # Gerador já existente no seu notebook
        # ----------------------------------------------------

        iterator = rows(
            file,
            cfg,
            union
        )

        total_lido = 0

        total_enviado = 0

        # ====================================================
        # PROCESSAR BATCHES
        # ====================================================

        for numero_batch, batch in enumerate(

            gerar_batches(
                iterator,
                batch_size
            ),

            start=1
        ):

            quantidade_batch = len(
                batch
            )

            total_lido += quantidade_batch

            # ------------------------------------------------
            # DataFrame
            # ------------------------------------------------

            df = preparar_dataframe(
                batch=batch,
                ano=ano,
                arquivo=caminho,
                digest=digest,
                stamp=stamp
            )

            # ------------------------------------------------
            # Garantir ordem das colunas
            # ------------------------------------------------

            colunas_esperadas = (
                list(union)
                +
                [
                    "ETL_ANO",
                    "ETL_ARQUIVO",
                    "ETL_SHA256",
                    "ETL_CARGA_UTC"
                ]
            )

            df = df[
                colunas_esperadas
            ]

            # ------------------------------------------------
            # Carregar no Snowflake
            # ------------------------------------------------

            nrows = escrever_dataframe_snowflake(
                conn=conn,
                df=df,
                temp_table=temp_table,
                chunk_size=batch_size
            )

            total_enviado += nrows

            print(
                f"Ano {ano} | "
                f"Batch {numero_batch:03d} | "
                f"Batch: {quantidade_batch:,} | "
                f"Acumulado: {total_enviado:,}"
            )

            # ------------------------------------------------
            # Liberar memória
            # ------------------------------------------------

            del df
            del batch

        # ====================================================
        # VALIDAR TEMP
        # ====================================================

        total_temp = contar_temp(
            conn,
            temp_table
        )

        print()
        print(
            f"Linhas lidas ........: {total_lido:,}"
        )

        print(
            f"Linhas enviadas .....: {total_enviado:,}"
        )

        print(
            f"Linhas na TEMP ......: {total_temp:,}"
        )

        if total_lido != total_temp:

            raise ValueError(
                f"""
ERRO DE VALIDAÇÃO

Arquivo: {caminho.name}

Lidas:
{total_lido:,}

Snowflake TEMP:
{total_temp:,}
"""
            )

        # ====================================================
        # VERIFICAR SE ARQUIVO FOI ALTERADO
        # ====================================================

        digest_final = checksum(
            caminho
        )

        if digest_final != digest:

            raise ValueError(
                f"""
O arquivo foi alterado durante a carga:

{caminho}
"""
            )

        # ====================================================
        # PUBLICAR NA STG
        # ====================================================

        print()
        print(
            "Validação OK."
        )

        print(
            "Publicando dados na tabela definitiva..."
        )

        publicar_ano(
            conn=conn,
            table_name=table_name,
            temp_table=temp_table,
            ano=ano
        )

        # ====================================================
        # VALIDAR TABELA DEFINITIVA
        # ====================================================

        total_final = validar_carga_final(
            conn=conn,
            table_name=table_name,
            ano=ano
        )

        if total_final != total_lido:

            raise ValueError(
                f"""
ERRO APÓS PUBLICAÇÃO

Arquivo:
{caminho.name}

Esperado:
{total_lido:,}

Snowflake:
{total_final:,}
"""
            )

        # ====================================================
        # LOG JSON
        # ====================================================

        output_path = Path(
            output
        )

        output_path.mkdir(
            parents=True,
            exist_ok=True
        )

        log = {

            "year": ano,

            "rows": total_final,

            "sha256": digest,

            "loaded_at": stamp.isoformat(),

            "file": str(caminho),

            "table": table_name,

            "database": SF_DATABASE,

            "schema": SF_SCHEMA,

            "warehouse": SF_WAREHOUSE
        }

        with (
            output_path / "loads_snowflake.jsonl"
        ).open(
            "a",
            encoding="utf-8"
        ) as stream:

            stream.write(
                json.dumps(
                    log,
                    ensure_ascii=False
                )
                +
                "\n"
            )

        print()
        print(
            "✅ CARGA CONCLUÍDA"
        )

        print(
            f"{table_name} | "
            f"{ano} | "
            f"{total_final:,} registros"
        )


# ============================================================
# 15. PILOTO
# ============================================================

EXECUTAR_PILOTO = True


if EXECUTAR_PILOTO:

    for cadastro, cfg in configuracoes.items():

        print()
        print("#" * 70)
        print(
            f"PILOTO: {cadastro}"
        )
        print("#" * 70)

        arquivos_piloto = [

            arquivo

            for arquivo
            in inventarios[cadastro]

            if arquivo["year"] == 2015
        ]

        if not arquivos_piloto:

            print(
                f"Nenhum arquivo de 2015 encontrado "
                f"para {cadastro}."
            )

            continue

        load_snowflake(

            conn=conn_sf,

            files=arquivos_piloto,

            cfg=cfg,

            union=schemas[cadastro],

            output=cfg["output_dir"],

            batch_size=50000
        )

else:

    print(
        """
Piloto não executado.

Depois de configurar o Snowflake,
altere:

EXECUTAR_PILOTO = True
"""
    )

Configurações do Snowflake carregadas.
Conectado ao Snowflake.

Conexão Snowflake OK

Database  : PROJETO_INEP
Schema    : STAGE
Warehouse : COMPUTE_WH
Role      : ACCOUNTADMIN
User      : HSILVA


######################################################################
PILOTO: CURSOS
######################################################################
Tabela verificada/criada: PROJETO_INEP.STAGE.STG_CURSOS

INICIANDO CARGA
Tabela : STG_CURSOS
Ano    : 2015
Arquivo: MICRODADOS_CADASTRO_CURSOS_2015.CSV
Ano 2015 | Batch 001 | Batch: 50,000 | Acumulado: 50,000
Ano 2015 | Batch 002 | Batch: 31,156 | Acumulado: 81,156

Linhas lidas ........: 81,156
Linhas enviadas .....: 81,156
Linhas na TEMP ......: 81,156

Validação OK.
Publicando dados na tabela definitiva...

✅ CARGA CONCLUÍDA
STG_CURSOS | 2015 | 81,156 registros

######################################################################
PILOTO: IES
######################################################################
Tabela verificada/cri

In [8]:
EXECUTAR_CARGA_COMPLETA = True


if EXECUTAR_CARGA_COMPLETA:

    for cadastro, cfg in configuracoes.items():

        print()
        print("#" * 70)
        print(f"CARGA COMPLETA: {cadastro}")
        print("#" * 70)

        load_snowflake(

            conn=conn_sf,

            files=inventarios[cadastro],

            cfg=cfg,

            union=schemas[cadastro],

            output=cfg["output_dir"],

            batch_size=50000
        )


######################################################################
CARGA COMPLETA: CURSOS
######################################################################
Tabela verificada/criada: PROJETO_INEP.STAGE.STG_CURSOS

INICIANDO CARGA
Tabela : STG_CURSOS
Ano    : 2015
Arquivo: MICRODADOS_CADASTRO_CURSOS_2015.CSV
Ano 2015 | Batch 001 | Batch: 50,000 | Acumulado: 50,000
Ano 2015 | Batch 002 | Batch: 31,156 | Acumulado: 81,156

Linhas lidas ........: 81,156
Linhas enviadas .....: 81,156
Linhas na TEMP ......: 81,156

Validação OK.
Publicando dados na tabela definitiva...

✅ CARGA CONCLUÍDA
STG_CURSOS | 2015 | 81,156 registros

INICIANDO CARGA
Tabela : STG_CURSOS
Ano    : 2016
Arquivo: MICRODADOS_CADASTRO_CURSOS_2016.CSV
Ano 2016 | Batch 001 | Batch: 50,000 | Acumulado: 50,000
Ano 2016 | Batch 002 | Batch: 42,866 | Acumulado: 92,866

Linhas lidas ........: 92,866
Linhas enviadas .....: 92,866
Linhas na TEMP ......: 92,866

Validação OK.
Publicando dados na tabela definitiva...

✅ CARGA